[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 03](README.md)

# OpenMP: bucles, reducciones y SIMD

**Tema:** 03 · **Sesiones:** 12, 13 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo distribuir iteraciones y combinar resultados sin introducir desbalance ni error numérico injustificado?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** Distribuir iteraciones solo ayuda si la carga queda balanceada, conserva localidad y combina resultados con la precisión acordada.

**Prerrequisitos.**

- Memoria compartida, carreras y sincronización.
- Compilación C/C++ con advertencias habilitadas.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Comparar schedules con una carga conocida.
- Distinguir reduction, atomic y critical.
- Medir error numérico además del tiempo.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Una reducción crea acumuladores privados y una combinación definida por el operador.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

Atomic protege una actualización compatible; critical serializa una región arbitraria.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

La suma en punto flotante no es asociativa y el orden paralelo puede cambiar el redondeo.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- fork–join — creación y reunión de trabajo paralelo
- entorno de datos — clasificación shared/private/firstprivate
- granularidad — cantidad de trabajo útil por unidad planificada


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Distribucion Trabajo

![Iteraciones distribuidas y reducción final](../../images/distribucion-trabajo.svg)

**Cómo leerlo.** Verifica dos propiedades: cada iteración pertenece a un trabajador y la combinación de parciales reproduce la referencia serial.

### Metodo Rendimiento

![Ciclo de medición, resumen, perfil e hipótesis](../../images/metodo-rendimiento.svg)

**Cómo leerlo.** Una medición se repite y resume antes de perfilar. La conclusión genera un experimento nuevo cambiando una sola variable controlada.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "03"
NOTEBOOK = "03_openmp/02_bucles_reducciones.ipynb"
assert (ROOT / "curso" / "notebooks" / "03_openmp" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Balance de carga

**Situación.** Se compara una asignación estática contigua con round-robin para costos crecientes.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
costs = [1 + (i % 7) ** 2 for i in range(32)]
threads = 4
contiguous = [sum(costs[t*8:(t+1)*8]) for t in range(threads)]
cyclic = [sum(costs[t::threads]) for t in range(threads)]
def imbalance(loads): return max(loads) / (sum(loads) / len(loads))
assert sum(contiguous) == sum(costs) == sum(cyclic)
print("contiguo", contiguous, "desbalance", round(imbalance(contiguous), 3))
print("cíclico ", cyclic, "desbalance", round(imbalance(cyclic), 3))


### Explicación del resultado

El mejor schedule depende del costo por iteración, localidad y overhead; la tabla formula una hipótesis, no una regla universal.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Precisión de la reducción

**Situación.** Se contrasta suma ingenua con `math.fsum` como referencia numérica más estable.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
import math
values = [1e16, 1.0, -1e16] * 1000
naive = sum(values)
stable = math.fsum(values)
print({"sum": naive, "fsum": stable, "error_absoluto": abs(naive - stable)})
assert stable == 1000.0


### Lectura razonada

El resultado esperado y la tolerancia deben definirse antes de comparar estrategias paralelas.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Qué cambiarías entre schedule estático y dinámico si cada iteración tuviera el mismo costo pero fuerte localidad?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Comparar `schedule(static)`, dinámico y guiado con la misma entrada.
2. Validar una integral/reducción frente a referencia.
3. Reportar tiempo, eficiencia y error para cada configuración.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Usar critical para toda la iteración.
- Aceptar igualdad exacta de flotantes sin análisis.
- Vectorizar un bucle con dependencias.


## Criterios de aceptación

- Cláusula de reducción correcta.
- Tolerancia y referencia documentadas.
- Schedule y chunk registrados con el resultado.


## Síntesis

- La pregunta que debes poder responder es: **¿Cómo distribuir iteraciones y combinar resultados sin introducir desbalance ni error numérico injustificado?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Integral OpenMP](../../../openmp/integral.cc)
- [Reducción](../../../openmp/reduction/integral.cc)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 03](README.md)
